In [10]:
import pandas as pd

df = pd.read_csv("../data/processed/analfabetismo_censos.csv")
df.info()
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 7920 entries, 0 to 7919
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   ano                 7920 non-null   int64  
 1   nivel               7920 non-null   str    
 2   territorio          7920 non-null   str    
 3   situacao            7920 non-null   str    
 4   sexo                7920 non-null   str    
 5   faixa_etaria        7920 non-null   str    
 6   taxa_analfabetismo  7920 non-null   float64
dtypes: float64(1), int64(1), str(5)
memory usage: 707.9 KB


,ano,nivel,territorio,situacao,sexo,faixa_etaria,taxa_analfabetismo
0,1991,UF,Acre,Rural,Total,15 a 17 anos,50.91
1,1991,UF,Acre,Rural,Total,18 e 19 anos,52.23
2,1991,UF,Acre,Rural,Total,20 a 24 anos,54.88
3,1991,UF,Acre,Rural,Total,25 a 34 anos,57.00
4,1991,UF,Acre,Rural,Total,35 a 44 anos,63.53


In [11]:
import plotly.express as px

brasil = df[
    (df["territorio"] == "Brasil")
    & (df["situacao"] == "Total")
    & (df["sexo"] == "Total")
    & (df["faixa_etaria"] == "Total")
]

fig = px.line(
    brasil, x="ano", y="taxa_analfabetismo", markers=True,
    title="Taxa de analfabetismo, população de 15 anos ou mais, Brasil",
    labels={"ano": "Ano do Censo", "taxa_analfabetismo": "Analfabetismo (%)"},
)
fig.update_xaxes(tickvals=[1991, 2000, 2010, 2022])
fig.show()

brasil[["ano", "taxa_analfabetismo"]].assign(queda=brasil["taxa_analfabetismo"].diff())

,ano,taxa_analfabetismo,queda
169,1991,20.07,NaN
1499,2000,13.63,-6.44
4139,2010,9.62,-4.01
4950,2022,7.00,-2.62


A partir dos dados, é possível identificar que a a maior queda ocorreu entre 1991 e 2000 (−6,44 pontos percentuais), seguida de 2000–2010 (−4,01) e 2010–2022 (−2,62). Já no Censo de 2022 a queda foi menos expressiva - entretanto, os intervalos de tempo não são iguais (9, 10 e 12 anos), podendo conduzir a avaliação para o favorecimento de intervalos mais longos. Para considerar este critério, comparo as quedas por ano a seguir.

In [12]:
brasil = brasil.assign(
    anos=brasil["ano"].diff(),
    queda=brasil["taxa_analfabetismo"].diff(),
)
brasil["queda_por_ano"] = (brasil["queda"] / brasil["anos"]).round(2)
brasil[["ano", "taxa_analfabetismo", "queda", "anos", "queda_por_ano"]]

,ano,taxa_analfabetismo,queda,anos,queda_por_ano
169,1991,20.07,NaN,NaN,NaN
1499,2000,13.63,-6.44,9.0,-0.72
4139,2010,9.62,-4.01,10.0,-0.40
4950,2022,7.00,-2.62,12.0,-0.22


Após a inclusão da coluna "queda_por_ano", o resultado fica de acordo com a primeira impressão: mesmo corrigindo pelo tamanho do intervalo, o ritmo anual de queda diminuiu a cada período. Para deixar a tabela ainda mais clara, coloco a seguir a visualização por período (não apenas do ano final).

In [13]:
periodos = brasil.dropna(subset=["queda"]).copy()
periodos["periodo"] = (periodos["ano"] - periodos["anos"]).astype(int).astype(str) + "–" + periodos["ano"].astype(str)
periodos[["periodo", "queda", "queda_por_ano"]]

,periodo,queda,queda_por_ano
1499,1991–2000,-6.44,-0.72
4139,2000–2010,-4.01,-0.40
4950,2010–2022,-2.62,-0.22


Os dados são consistentes com a desaceleração da taxa de analfabetismo no último ciclo de avaliação do Censo. Entretanto, apenas este dado por si só não pode ser utilizado como causa de forma isolada, visto que quando a taxa já é mais baixa, pode haver dificuldade em reduzí-la ainda mais com o mesmo grau de expressão. Ainda, os dados não explicam a causa da queda - como a relação com determinadas políticas públicas ou programas.

In [14]:
ufs = df[
    (df["nivel"] == "UF")
    & (df["situacao"] == "Total")
    & (df["sexo"] == "Total")
    & (df["faixa_etaria"] == "Total")
]

resumo = ufs.groupby("ano")["taxa_analfabetismo"].agg(["min", "median", "max"]).round(2)
resumo["amplitude"] = (resumo["max"] - resumo["min"]).round(2)
resumo

,min,median,max,amplitude
ano,,,,
1991,9.24,20.62,45.32,36.08
2000,5.68,13.49,33.39,27.71
2010,3.47,9.85,24.32,20.85
2022,2.67,6.92,17.66,14.99


In [15]:
fig = px.box(
    ufs, x="ano", y="taxa_analfabetismo", points="all", hover_name="territorio",
    title="Analfabetismo nas 27 UFs, população de 15 anos ou mais",
    labels={"ano": "Ano do Censo", "taxa_analfabetismo": "Analfabetismo (%)"},
)
fig.update_xaxes(type="category")
fig.show()

A partir da tabela e do gráfico, já foi possível identificar que Alagoas foi o estado (UF) com maior analfabetismo nos quatro censos, enquanto que Distrito Federal e Santa Catarina foram os estados com menores índices apresentados. Para facilitar a leitura, incluo mais uma forma de visualização a seguir.

In [16]:
ranking = ufs.sort_values(["ano", "taxa_analfabetismo"])
extremos = pd.concat([ranking.groupby("ano").head(3), ranking.groupby("ano").tail(3)])
extremos.sort_values(["ano", "taxa_analfabetismo"])[["ano", "territorio", "taxa_analfabetismo"]]

,ano,territorio,taxa_analfabetismo
259,1991,Distrito Federal,9.24
739,1991,Rio de Janeiro,9.72
829,1991,Santa Catarina,9.91
649,1991,Piauí,41.67
559,1991,Paraíba,41.72
49,1991,Alagoas,45.32
1769,2000,Distrito Federal,5.68
3479,2000,Santa Catarina,6.32
3209,2000,Rio de Janeiro,6.64
2669,2000,Paraíba,29.71


In [17]:
resumo["razao_max_min"] = (resumo["max"] / resumo["min"]).round(1)
resumo

,min,median,max,amplitude,razao_max_min
ano,,,,,
1991,9.24,20.62,45.32,36.08,4.9
2000,5.68,13.49,33.39,27.71,5.9
2010,3.47,9.85,24.32,20.85,7.0
2022,2.67,6.92,17.66,14.99,6.6


In [18]:
((resumo.loc[2022, ["min", "median", "max"]] / resumo.loc[1991, ["min", "median", "max"]] - 1) * 100).round(1)

min      -71.1
median   -66.4
max      -61.0
dtype: float64

In [19]:
regioes = df[
    (df["nivel"] == "Região")
    & (df["situacao"] == "Total")
    & (df["sexo"] == "Total")
    & (df["faixa_etaria"] == "Total")
].pivot(index="territorio", columns="ano", values="taxa_analfabetismo")
regioes

ano,1991,2000,2010,2022
territorio,,,,
Centro-Oeste,16.75,10.76,7.19,5.06
Nordeste,37.65,26.20,19.06,14.21
Norte,24.64,16.34,11.18,8.16
Sudeste,12.31,8.13,5.45,3.92
Sul,11.85,7.66,5.10,3.45


Por meio da estratificação dos dados e avaliação por medidas de amplitude e razão, percebe-se que a desigualdade entre UFs diminuiu conforme a amplitude, mas não segundo a razão:

- Amplitude (maior − menor, em pontos percentuais): mede a desigualdade absoluta, ou seja, quantos pontos separam a pior UF da melhor.
- Razão (maior ÷ menor): mede a desigualdade relativa, ou seja, quantas vezes o pior é maior que o melhor.

**1. O que as duas medidas mostram:** a amplitude caiu de 36,08 para 14,99 pontos; a razão foi de 4,9 para 6,6 (com pico de 7,0 em 2010). 

**2. Por que mínima, mediana e máxima divergem:** de 1991 a 2022, o menor valor observado caiu 71,1%, a mediana 66,4% e o maior 61,0%. Como o menor valor observado caiu proporcionalmente mais que o maior (o mínimo foi do DF até 2010 e de SC em 2022; o máximo foi de Alagoas nos quatro censos), é por isso que a razão (maior ÷ menor) subiu mesmo com a amplitude caindo.

**3. Uma ressalva:** a razão é sensível quando o menor valor é muito baixo. Como o menor valor fica no denominador, quando ele é muito pequeno, uma variação mínima nele altera bastante a razão. Por exemplo, no ano de 2022, o menor valor foi de 2,67, enquanto o maior valor foi de 17,66, justificando o resultado da razão (6,6). Se o menor valor fosse ligeiramente maior - como 3,00 - a amplitude vai de 14,99 para 14,66 (reduzindo apenas 0,33 ponto), enquanto a razão vai de 6,6 para 5,9 (0,7).
Assim, se compararmos os períodos de 2010 (7,0) com 2022 (6,6), a diferença é de 0,4. Uma pequena mudança no menor valor já poderia eliminar essa diferença.
Ao comparar os períodos de 1991 (4,9) com 2022 (6,6), a diferença já é mais expressiva (1,7), ressaltando o aumento da desigualdade relativa no longo prazo.

**4. Comparação entre regiões:** a análise descreve a desigualdade entre os estados e ressalta a diferença entre regiões. Os estados com menores taxas de analfabetismo estão nas regiões Centro-Oeste, Sudeste e Sul, enquanto os estados com maiores taxas prevaleceram na região Nordeste. Comparando Nordeste e Sul:

| Ano | Nordeste | Sul | Diferença | Razão |
|---|---|---|---|---|
| 1991 | 37,65% | 11,85% | 25,8 pontos | 3,2 vezes |
| 2022 | 14,21% | 3,45% | 10,8 pontos | 4,1 vezes |

Com a queda da diferença, a razão de desigualdade sobe, o mesmo padrão observado entre as UFs.

**5. O que os dados não dizem:** a análise descreve as diferenças entre os estados e as regiões, mas não explica as causas (como a relação com políticas públicas, renda, acesso à escola, entre outras variáveis).

In [20]:
ordem = ["15 a 17 anos", "18 e 19 anos", "20 a 24 anos", "25 a 34 anos",
         "35 a 44 anos", "45 a 54 anos", "55 a 64 anos", "65 anos ou mais"]

idade = df[
    (df["territorio"] == "Brasil")
    & (df["situacao"] == "Total")
    & (df["sexo"] == "Total")
    & (df["faixa_etaria"].isin(ordem))
]

fig = px.line(
    idade, x="faixa_etaria", y="taxa_analfabetismo", color="ano", markers=True,
    category_orders={"faixa_etaria": ordem},
    title="Analfabetismo por faixa de idade, Brasil",
    labels={"faixa_etaria": "Faixa de idade", "taxa_analfabetismo": "Analfabetismo (%)", "ano": "Censo"},
)
fig.show()

In [21]:
idade.pivot(index="faixa_etaria", columns="ano", values="taxa_analfabetismo").loc[ordem].round(2)

ano,1991,2000,2010,2022
faixa_etaria,,,,
15 a 17 anos,12.42,4.85,2.20,1.59
18 e 19 anos,11.47,5.21,2.20,1.46
20 a 24 anos,12.18,6.73,2.77,1.54
25 a 34 anos,13.29,8.83,4.87,1.96
35 a 44 anos,19.16,11.54,8.30,4.11
45 a 54 anos,27.91,17.76,11.32,7.83
55 a 64 anos,35.15,27.18,17.78,11.47
65 anos ou mais,47.80,37.98,29.36,20.25


**Observação:** conforme a estratificação por faixa de idade, fica clara a maior taxa de analfabetismo entre pessoas com 65 anos ou mais. De acordo com análises anteriores, na avaliação por faixa etária também identifica-se a similaridade dos dados entre os anos de 2010 e 2022, especialmente nas faixas de 15 a 17 anos (2,2% em 2010 e 1,6% em 2022), 18 e 19 anos (2,2% em 2010 e 1,5% em 2022) e 20 a 24 anos (2,8% em 2010 e 1,5% em 2022). Como as taxas dessas faixas já são baixas (abaixo de 3%), há pouco espaço para quedas mais expressivas.
Comparando dados dos censos de 1991 x 2022, na avaliação da faixa de 15 a 17 anos, percebe-se redução expressiva (de 12,42% em 1991 para 1,59% em 2022), assim como nas demais faixas etárias, mas ainda com o mesmo padrão - a prevalência de analfabetismo no grupo de 65 anos ou mais (de 47,8% em 1991 para 20,3% em 2022). Em pontos percentuais, a maior queda foi entre os mais velhos, embora em proporção a maior queda tenha sido entre os mais jovens.

**Hipótese:** apesar de não podermos afirmar que os dados acompanham as mesmas pessoas ao longo do tempo, podemos considerar algumas hipóteses:

a. As gerações mais velhas, com menor escolaridade, deixam a população ao longo do tempo e são substituídas por gerações mais jovens, com mais escolaridade.
b. Os adultos que não foram alfabetizados tendem a seguir sem estudo. 

**Recomendação:** sugere-se a avaliação de programas e iniciativas para o ensino de pessoas adultas, como forma de analisar as causas e definir um plano de ação. 
